In [1]:
!pip install -qU langchain langchain-openai langchain-community langchain-experimental neo4j tiktoken ragas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-milvus 0.1.8 requires langchain-core<0.4,>=0.2.38, but you have langchain-core 1.4.8 which is incompatible.
langchain-chroma 0.2.2 requires langchain-core!=0.3.0,!=0.3.1,!=0.3.10,!=0.3.11,!=0.3.12,!=0.3.13,!=0.3.14,!=0.3.2,!=0.3.3,!=0.3.4,!=0.3.5,!=0.3.6,!=0.3.7,!=0.3.8,!=0.3.9,<0.4.0,>=0.2.43, but you have langchain-core 1.4.8 which is incompatible.
langchain-azure-ai 0.1.2 requires langchain-core<0.4.0,>=0.3.0, but you have langchain-core 1.4.8 which is incompatible.
langchain-azure-ai 0.1.2 requires langchain-openai<0.4.0,>=0.3.0, but you have langchain-openai 1.3.3 which is incompatible.
langchain-google-genai 2.1.0 requires langchain-core<0.4.0,>=0.3.43, but you have langchain-core 1.4.8 which is incompatible.
langchain-cohere 0.4.3 requires langchain-community<0.4.0,>=0.3.0, but you have langchain-

In [2]:
import os
import getpass
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# API 키 설정 (필요 시)
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

In [3]:
print("=== [연구자용 RAG] 내부 논문 DB (Vector) 초기화 ===")
# 대학원 연구실의 내부(과거) 데이터베이스라고 가정합니다.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = InMemoryVectorStore(embeddings)
vectorstore.add_texts([
    "[내부 DB - 2023년 보고서] 신약 'NovaX'의 임상 2상 성공률은 45%로 기록되었다.",
    "[내부 DB - 2022년 보고서] 'NovaX'의 주요 부작용은 가벼운 두통이다."
])
internal_retriever = vectorstore.as_retriever()

@tool
def search_internal_papers(query: str) -> str:
    """연구실 내부 Vector DB를 검색합니다. 과거의 연구 기록이나 임상 데이터를 찾을 때 사용하세요."""
    docs = internal_retriever.invoke(query)
    return "\n".join([doc.page_content for doc in docs])

=== [연구자용 RAG] 내부 논문 DB (Vector) 초기화 ===


In [4]:
# ==========================================
# 👩‍💻 [개별 실습 과제 (TODO) - 완성본]
# ==========================================
print("\n=== 📝 외부 Web Search 연동 및 교차 검증 에이전트 구축 ===")
print("연구 상황: 내부 DB는 2023년까지의 데이터만 있습니다.")
print("에이전트가 내부 DB를 먼저 찾고, 정보가 부족하거나 최신 정보가 필요하면 외부 검색(Mock)을 수행한 뒤,")
print("수치 연산 도구를 이용해 최종 통계값을 계산하도록 만들어야 합니다.\n")


# TODO 1: 외부 검색 도구(Mock) 생성하기
# 실제 Web Search API(Tavily 등) 대신, 'NovaX' + ('3상' 또는 '최신') 키워드가 들어오면
# 2026년 최신 뉴스를 반환하는 Mock 도구.
@tool
def search_external_web(query: str) -> str:
    """최신 논문이나 외부 뉴스를 웹에서 검색할 때 사용합니다.
    내부 DB에 없는 2024년 이후의 최신 임상 결과나 뉴스를 교차 검증할 때 호출하세요."""
    # 'NovaX'와 함께 '3상' 또는 '최신' 의도가 담긴 질의면 최신 뉴스를 반환
    if "NovaX" in query and ("3상" in query or "최신" in query):
        return (
            "[웹 검색 - 2026년 뉴스] NovaX 임상 3상 완료, "
            "최종 성공률 68% 달성, 투여 환자 수 5000명"
        )
    return "[웹 검색] 관련된 최신 외부 정보를 찾지 못했습니다."


# TODO 2: 임상 데이터 통계 계산 도구 생성 (Type-Hinting 엄격 적용)
# 환자 수(int)와 성공률(float, 예: 0.68)을 받아 완치된 예상 환자 수를 int 로 반환.
@tool
def calculate_cured_patients(patient_count: int, success_rate: float) -> int:
    """임상 시험의 예상 완치 환자 수를 계산합니다.

    Args:
        patient_count: 임상에 투여된 전체 환자 수입니다. 정수(int)로 전달하세요. (예: 5000)
        success_rate: 임상 성공률입니다. 0.0 ~ 1.0 사이의 소수(float)로 전달하세요.
            퍼센트(예: 68)가 아니라 비율(예: 0.68)이어야 합니다.

    Returns:
        완치된 것으로 예상되는 환자 수입니다. 소수점은 버리고 정수(int)로 반환합니다.
    """
    return int(patient_count * success_rate)


# TODO 3: 에이전트 시스템 프롬프트 작성 및 에이전트 생성
tools = [search_internal_papers, search_external_web, calculate_cured_patients]
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

system_prompt = """당신은 신약 임상 데이터를 분석하는 꼼꼼한 연구 보조 에이전트입니다.
다음 절차를 반드시 순서대로 지켜서 답변하세요.

1. 어떤 질문이든 먼저 `search_internal_papers` 도구로 연구실 내부 DB를 검색합니다.
2. 내부 데이터가 2024년 이전(과거) 자료이거나 최신 결과(예: 임상 3상)가 필요하면,
   반드시 `search_external_web` 도구로 최신 데이터를 검색해 교차 검증합니다.
3. 환자 수와 성공률을 바탕으로 한 수치 계산이 필요하면, 직접 계산하지 말고
   반드시 `calculate_cured_patients` 도구를 사용합니다.
   - 이때 success_rate 는 퍼센트(68)가 아니라 비율(0.68)로 변환해 전달해야 합니다.
4. 마지막에 내부 자료와 외부 최신 자료를 비교하고, 계산 결과를 포함해
   한국어로 명확하게 요약합니다."""

agent_executor = create_react_agent(llm, tools, prompt=system_prompt)


# TODO 4: 복합 추론 질문 실행
question = (
    "우리 연구실에 기록된 NovaX의 임상 성공률을 확인해보고, "
    "최신 웹 검색을 통해 3상 결과가 나왔는지 교차 검증해줘. "
    "그리고 3상 결과의 환자수와 성공률을 바탕으로 완치된 예상 환자 수를 계산해줘."
)
res = agent_executor.invoke({"messages": [("user", question)]})

# 에이전트가 어떤 순서로 도구를 호출했는지 로그 확인
print("=== 🛠️ 도구 호출 순서 ===")
for msg in res["messages"]:
    if msg.type == "ai" and msg.tool_calls:
        for t in msg.tool_calls:
            print(f"  - {t['name']}  args={t['args']}")

print("\n=== ✅ 최종 답변 ===")
print(res["messages"][-1].content)


=== 📝 외부 Web Search 연동 및 교차 검증 에이전트 구축 ===
연구 상황: 내부 DB는 2023년까지의 데이터만 있습니다.
에이전트가 내부 DB를 먼저 찾고, 정보가 부족하거나 최신 정보가 필요하면 외부 검색(Mock)을 수행한 뒤,
수치 연산 도구를 이용해 최종 통계값을 계산하도록 만들어야 합니다.



/var/folders/dy/kygc74fj3xjckz0h1d1z4cdc0000gn/T/ipykernel_63001/1391161036.py:59: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools, prompt=system_prompt)


=== 🛠️ 도구 호출 순서 ===
  - search_internal_papers  args={'query': 'NovaX 임상 성공률'}
  - search_external_web  args={'query': 'NovaX 임상 3상 결과 2024'}
  - calculate_cured_patients  args={'patient_count': 5000, 'success_rate': 0.68}

=== ✅ 최종 답변 ===
우리 연구실의 기록에 따르면, NovaX의 임상 2상 성공률은 45%로 나타났습니다. 그러나 최신 웹 검색 결과에 따르면, NovaX의 임상 3상에서 최종 성공률은 68%로 확인되었으며, 총 5000명의 환자가 투여되었습니다.

임상 3상의 성공률을 바탕으로 계산한 결과, 예상 완치된 환자 수는 3400명입니다.

요약하자면:
- NovaX 임상 2상 성공률: 45%
- NovaX 임상 3상 성공률: 68%
- 임상 3상 환자 수: 5000명
- 예상 완치된 환자 수: 3400명
